In [1]:
# Set project root
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

while PROJECT_ROOT.name != "archivist" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if PROJECT_ROOT.name != "archivist":
    raise RuntimeError("Could not find Archivist project root")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /home/kali/workspace/github.com/tlazizbek/archivist


In [2]:
# Import libraries
import json
import sqlite3
import pandas as pd
import numpy as np

In [3]:
# Use the correct database
DB_PATH = PROJECT_ROOT / "archivist.db"

print("Database:", DB_PATH)
print("Exists:", DB_PATH.exists())

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

Database: /home/kali/workspace/github.com/tlazizbek/archivist/archivist.db
Exists: True


In [4]:
with sqlite3.connect(DB_PATH) as connection:
    tables = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """,
        connection,
    )

tables

,name
0,chunks
1,documents
2,query_feedback
3,query_logs
4,sqlite_sequence


In [5]:
with sqlite3.connect(DB_PATH) as connection:
    query_logs = pd.read_sql_query(
        """
        SELECT
            id,
            query_text,
            retrieval_method,
            retrieved_chunk_ids,
            answer_text,
            latency_ms,
            llm_model,
            created_at
        FROM query_logs
        ORDER BY id
        """,
        connection,
    )

print("Rows:", len(query_logs))
query_logs.head()

Rows: 4


,id,query_text,retrieval_method,retrieved_chunk_ids,answer_text,latency_ms,llm_model,created_at
0,1,How can I become part of an organization?,hybrid,"""[5, 6, 17, 20, 18]""","\n\nBased on the provided context, you can bec...",28248,openrouter/free,2026-08-25 10:02:59
1,2,What is mentioned in the Day 20 ingestion test?,hybrid,"""[21, 8, 9, 11, 16]""",The Day 20 ingestion test mentions a script to...,24292,openrouter/free,2026-08-25 10:29:03
2,3,What is the verification code for the Archivis...,hybrid,"""[22, 13, 14, 21, 16]""",User Safety: safe,26753,openrouter/free,2026-08-25 10:31:04
3,4,What is the verification code in the Archivist...,hybrid,"""[23, 22, 21, 13, 14]""",The verification code in the Archivist Day 20 ...,24863,openrouter/free,2026-08-25 10:33:50


In [6]:
with sqlite3.connect(DB_PATH) as connection:
    corpus_stats = pd.read_sql_query(
        """
        SELECT
            d.id AS document_id,
            d.title,
            COUNT(c.id) AS chunk_count,
            COALESCE(SUM(LENGTH(c.content)), 0) AS character_count
        FROM documents d
        LEFT JOIN chunks c
            ON c.document_id = d.id
        GROUP BY d.id, d.title
        ORDER BY d.id
        """,
        connection,
    )

print("Corpus stats:", corpus_stats.shape)

corpus_stats.head()

Corpus stats: (17, 4)


,document_id,title,chunk_count,character_count
0,1,personal-profile,2,4675
1,2,username-changes,2,5119
2,3,organization-membership,1,3495
3,4,organization-profile,1,2150
4,5,text_1,1,674


In [7]:
query_logs["created_at"] = pd.to_datetime(query_logs["created_at"])

queries_per_day = (
    query_logs
    .assign(date=query_logs["created_at"].dt.date)
    .groupby("date")
    .size()
    .reset_index(name="query_count")
)

queries_per_day

,date,query_count
0,2026-08-25,4


In [8]:
average_latency_ms = query_logs["latency_ms"].mean()

print(f"Average latency: {average_latency_ms:.2f} ms")

Average latency: 26039.00 ms


In [10]:
p95_latency_ms = query_logs["latency_ms"].quantile(0.95)

print(f"P95 latency: {p95_latency_ms:.2f} ms")

P95 latency: 28023.75 ms


In [11]:
retrieval_method_usage = (
    query_logs["retrieval_method"]
    .value_counts()
    .rename_axis("retrieval_method")
    .reset_index(name="query_count")
)

retrieval_method_usage["percentage"] = (
    retrieval_method_usage["query_count"]
    / retrieval_method_usage["query_count"].sum()
    * 100
)

retrieval_method_usage

,retrieval_method,query_count,percentage
0,hybrid,4,100.0


In [12]:
query_logs["retrieved_chunk_ids"] = query_logs["retrieved_chunk_ids"].apply(
    lambda value: json.loads(value) if isinstance(value, str) else value
)

query_logs[["id", "retrieved_chunk_ids"]]

,id,retrieved_chunk_ids
0,1,"[5, 6, 17, 20, 18]"
1,2,"[21, 8, 9, 11, 16]"
2,3,"[22, 13, 14, 21, 16]"
3,4,"[23, 22, 21, 13, 14]"


In [ ]:
retrieved_chunks = (
    query_logs[
        ["id", "retrieved_chunk_ids"]
    ]
    .explode("retrieved_chunk_ids")
    .rename(
        columns={
            "id": "query_id",
            "retrieved_chunk_ids": "chunk_id",
        }
    )
)

retrieved_chunks["chunk_id"] = retrieved_chunks["chunk_id"].astype(int)

retrieved_chunks.head(10)

In [14]:
with sqlite3.connect(DB_PATH) as connection:
    chunk_documents = pd.read_sql_query(
        """
        SELECT
            c.id AS chunk_id,
            c.document_id,
            d.title
        FROM chunks c
        JOIN documents d
            ON d.id = c.document_id
        """,
        connection,
    )

chunk_documents.head()

,chunk_id,document_id,title
0,1,1,personal-profile
1,2,1,personal-profile
2,3,2,username-changes
3,4,2,username-changes
4,5,3,organization-membership


In [15]:
retrieved_documents = retrieved_chunks.merge(
    chunk_documents,
    on="chunk_id",
    how="left",
)

most_retrieved_documents = (
    retrieved_documents
    .groupby(["document_id", "title"])
    .size()
    .reset_index(name="retrieval_count")
    .sort_values("retrieval_count", ascending=False)
    .reset_index(drop=True)
)

most_retrieved_documents

,document_id,title,retrieval_count
0,10,email-addresses,4
1,15,README,3
2,6,personal-dashboard-quickstart,2
3,16,README,2
4,12,README,2
5,13,account-management,2
6,3,organization-membership,1
7,4,organization-profile,1
8,8,text_4,1
9,14,contributions-on-your-profile,1


In [16]:
powerbi_queries = query_logs[
    [
        "id",
        "query_text",
        "retrieval_method",
        "latency_ms",
        "llm_model",
        "created_at",
    ]
].copy()

powerbi_queries["date"] = powerbi_queries["created_at"].dt.date

powerbi_queries

,id,query_text,retrieval_method,latency_ms,llm_model,created_at,date
0,1,How can I become part of an organization?,hybrid,28248,openrouter/free,2026-08-25 10:02:59,2026-08-25
1,2,What is mentioned in the Day 20 ingestion test?,hybrid,24292,openrouter/free,2026-08-25 10:29:03,2026-08-25
2,3,What is the verification code for the Archivis...,hybrid,26753,openrouter/free,2026-08-25 10:31:04,2026-08-25
3,4,What is the verification code in the Archivist...,hybrid,24863,openrouter/free,2026-08-25 10:33:50,2026-08-25


In [17]:
powerbi_documents = most_retrieved_documents.copy()

powerbi_documents

,document_id,title,retrieval_count
0,10,email-addresses,4
1,15,README,3
2,6,personal-dashboard-quickstart,2
3,16,README,2
4,12,README,2
5,13,account-management,2
6,3,organization-membership,1
7,4,organization-profile,1
8,8,text_4,1
9,14,contributions-on-your-profile,1


In [18]:
OUTPUT_DIR = PROJECT_ROOT / "analytics" / "exports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

queries_csv = OUTPUT_DIR / "query_analytics.csv"
documents_csv = OUTPUT_DIR / "document_retrievals.csv"

powerbi_queries.to_csv(queries_csv, index=False)
powerbi_documents.to_csv(documents_csv, index=False)

print("Exported:")
print(queries_csv)
print(documents_csv)

Exported:
/home/kali/workspace/github.com/tlazizbek/archivist/analytics/exports/query_analytics.csv
/home/kali/workspace/github.com/tlazizbek/archivist/analytics/exports/document_retrievals.csv


In [19]:
print("Query analytics:")
print(pd.read_csv(queries_csv).head())

print("\nDocument retrievals:")
print(pd.read_csv(documents_csv).head())

Query analytics:
   id                                         query_text retrieval_method  \
0   1          How can I become part of an organization?           hybrid   
1   2    What is mentioned in the Day 20 ingestion test?           hybrid   
2   3  What is the verification code for the Archivis...           hybrid   
3   4  What is the verification code in the Archivist...           hybrid   

   latency_ms        llm_model           created_at        date  
0       28248  openrouter/free  2026-08-25 10:02:59  2026-08-25  
1       24292  openrouter/free  2026-08-25 10:29:03  2026-08-25  
2       26753  openrouter/free  2026-08-25 10:31:04  2026-08-25  
3       24863  openrouter/free  2026-08-25 10:33:50  2026-08-25  

Document retrievals:
   document_id                          title  retrieval_count
0           10                email-addresses                4
1           15                         README                3
2            6  personal-dashboard-quickstart           

In [20]:
assert not query_logs.empty, "query_logs is empty"
assert not corpus_stats.empty, "corpus_stats is empty"
assert not queries_per_day.empty, "queries_per_day is empty"
assert not retrieval_method_usage.empty, "retrieval_method_usage is empty"
assert not most_retrieved_documents.empty, "most_retrieved_documents is empty"
assert queries_csv.exists(), "Query CSV was not created"
assert documents_csv.exists(), "Document CSV was not created"

print("Day 21 analysis completed successfully.")
print()
print(f"Queries: {len(query_logs)}")
print(f"Documents: {len(corpus_stats)}")
print(f"Average latency: {average_latency_ms:.2f} ms")
print(f"P95 latency: {p95_latency_ms:.2f} ms")
print(f"Query CSV: {queries_csv}")
print(f"Document CSV: {documents_csv}")

Day 21 analysis completed successfully.

Queries: 4
Documents: 17
Average latency: 26039.00 ms
P95 latency: 28023.75 ms
Query CSV: /home/kali/workspace/github.com/tlazizbek/archivist/analytics/exports/query_analytics.csv
Document CSV: /home/kali/workspace/github.com/tlazizbek/archivist/analytics/exports/document_retrievals.csv
